# Belief Revision

**Last updated:** 2026-04-09 12:00 PT

**Track:** Learning | **Function:** Evidence integration & updating

Can the model update its answer when given new evidence that changes the correct conclusion?
Tests whether self-generated notes about what changed help overcome anchoring bias.

**Difficulty grid:** reasoning depth × contradiction strength × 3 seeds = 27 items


In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import random
import re
import json
import time

print('Available models:', list(kbench.llms.keys()))
print('Default model:', kbench.llm)

In [ ]:
def strip_thinking(text: str) -> str:
    """Remove <think>...</think> blocks from reasoning model output."""
    if '</think>' in text:
        return text.split('</think>', 1)[1].strip()
    return text.strip()

def extract_short_answer(response: str) -> str:
    """Extract a short answer (last short line, stripped of formatting)."""
    text = strip_thinking(response)
    text = re.sub(r'[`*_]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    if not lines: return ''
    for line in reversed(lines):
        clean = line.strip('"\'\'\u201c\u201d').rstrip('.!?')
        if len(clean) <= 30:
            return clean.lower().strip()
    return lines[-1].strip('"\'\'\u201c\u201d').rstrip('.!?').lower().strip()

def extract_number(response: str) -> str:
    """Extract a number from model response."""
    text = strip_thinking(response)
    text = re.sub(r'[`*_]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    for line in reversed(lines):
        clean = line.strip('.!?, ')
        if re.match(r'^-?\d+$', clean): return clean
    nums = re.findall(r'-?\d+', text)
    return nums[-1] if nums else ''

def extract_category(response: str) -> str:
    """Extract a category letter (A-D) from model response."""
    text = strip_thinking(response)
    text = re.sub(r'[`*_]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    for line in reversed(lines):
        m = re.search(r'\b([A-D])\b', line)
        if m and len(line) < 30: return m.group(1)
    for line in reversed(lines):
        m = re.search(r'([A-D])', line)
        if m: return m.group(1)
    return ''


In [ ]:
# Constants used in results analysis
BELIEF_FNS_KEYS = ['simple', 'chain', 'nested']
STRENGTHS = ['weak', 'moderate', 'strong']
SEEDS = 3

In [ ]:
# === Load dataset from Kaggle ===
import os, glob
candidates = glob.glob('/kaggle/input/**/belief_revision_dataset.csv', recursive=True)
if candidates:
    csv_path = candidates[0]
else:
    csv_path = '/kaggle/input/belief_revision_dataset.csv'
print(f'Loading from: {csv_path}')
dataset = pd.read_csv(csv_path)
print(f'Dataset: {len(dataset)} items')

In [ ]:
PRE_P = """Consider ALL of the following evidence:

{material}

Based on ALL the evidence (including any corrections or updates), what is the answer?

Reply with ONLY the answer (a name, letter, or number). Nothing else."""

STUDY_P = """Consider the following evidence that includes an update/correction:

{material}

Analyze what changed:
1. What did the initial evidence suggest?
2. What does the new evidence change?
3. How does this affect the conclusion?
4. What is the correct answer considering ALL evidence?

Write a clear analysis."""

POST_P = """You previously analyzed some evidence with an update and wrote these notes:

--- YOUR NOTES ---
{notes}
--- END NOTES ---

Here is the full evidence again:
{material}

Based on ALL the evidence and your analysis, what is the answer?

Reply with ONLY the answer (a name, letter, or number). Nothing else."""


## Evaluation


In [ ]:
all_results = []

@kbench.task(name='belief_revision',
             description='Pre/post learning belief revision with contradicting evidence')
def belief_revision(llm) -> float:
    model_name = str(llm)
    correct = 0
    total = len(dataset)

    for _, row in dataset.iterrows():
        material = row['material']
        test_input = row['test_input']
        expected = row['expected']
        complexity = row['complexity']
        evidence = row['evidence']
        difficulty_label = row['difficulty_label']
        seed = int(row['seed'])
        item_desc = row['item_desc']
        ts = time.strftime('%Y-%m-%d %H:%M:%S')
        tid = f'{complexity}_{evidence}_{seed}'

        t0 = time.time()
        pre_raw = llm.prompt(PRE_P.format(material=material))
        pre_time = time.time() - t0
        pre_extracted = extract_short_answer(pre_raw)
        pre_correct = pre_extracted == expected

        t0 = time.time()
        study_raw = llm.prompt(STUDY_P.format(material=material))
        study_time = time.time() - t0
        notes = strip_thinking(study_raw)

        t0 = time.time()
        post_raw = llm.prompt(POST_P.format(notes=notes, material=material))
        post_time = time.time() - t0
        post_extracted = extract_short_answer(post_raw)
        post_correct = post_extracted == expected

        if post_correct:
            correct += 1

        all_results.append({
            'timestamp': ts, 'model': model_name, 'task_id': tid,
            'complexity': complexity, 'evidence': evidence, 'difficulty_label': difficulty_label,
            'seed': int(seed), 'item_desc': item_desc,
            'test_input': test_input, 'expected': expected,
            'pre_raw': pre_raw, 'pre_extracted': pre_extracted, 'pre_correct': int(pre_correct),
            'study_notes': notes,
            'post_raw': post_raw, 'post_extracted': post_extracted, 'post_correct': int(post_correct),
            'pre_time_s': pre_time, 'study_time_s': study_time, 'post_time_s': post_time
        })

        b,l = 'Y' if pre_correct else 'N', 'Y' if post_correct else 'N'
        print(f'[{model_name:40s}] [{difficulty_label:16s}] expected="{expected}"  pre="{pre_extracted}"({b})  post="{post_extracted}"({l})')
        # Per-item assertion for diagnostics
        kbench.assertions.assert_equal(post_extracted, expected)

    score = correct / total
    print(f'\nScore: {correct}/{total} = {score:.4f}')
    return score

try:
    runs = belief_revision.evaluate(llm=[kbench.llm],
                                    n_jobs=1, timeout=3600, max_attempts=1)
    print(f'\nDone: {len(runs.as_dataframe())} items')
except Exception as e:
    print(f'SDK post-processing error (non-fatal): {e}')

## Results


In [ ]:
results = pd.DataFrame(all_results)
print(f'Total runs: {len(results)}\n')

pre_acc = results['pre_correct'].mean()
post_acc = results['post_correct'].mean()
gain = post_acc - pre_acc

print('=' * 60)
print(f'Model: {results["model"].iloc[0] if len(results) > 0 else "N/A"}')
print(f'Pre-learning accuracy:  {pre_acc:.1%}')
print(f'Post-learning accuracy: {post_acc:.1%}')
print(f'Learning gain:          {gain:+.1%}')
print()

print('By Difficulty:')
print('-' * 60)
for label in sorted(results['difficulty_label'].unique()):
    sub = results[results['difficulty_label'] == label]
    b = sub['pre_correct'].mean()
    l = sub['post_correct'].mean()
    g = l - b
    print(f'  {label:25s}  pre={b:.1%}  post={l:.1%}  gain={g:+.1%}  (n={len(sub)})')

print()
print(f'Timing: pre={results["pre_time_s"].mean():.1f}s  study={results["study_time_s"].mean():.1f}s  post={results["post_time_s"].mean():.1f}s')

In [ ]:
print('STUDY NOTES INSPECTION')
print('=' * 60)
for _, row in results.iterrows():
    b = 'Y' if row['pre_correct'] else 'N'
    l = 'Y' if row['post_correct'] else 'N'
    print(f'\n--- [{row["difficulty_label"]}] seed={row["seed"]} ---')
    print(f'Item: {row["item_desc"]}')
    print(f'Expected: "{row["expected"]}"')
    print(f'Pre: "{row["pre_extracted"]}" ({b})  Post: "{row["post_extracted"]}" ({l})')
    print(f'Notes (first 300 chars): {row["study_notes"][:300]}')
    print('...' if len(str(row['study_notes'])) > 300 else '')


In [ ]:
!pip install -q matplotlib
import matplotlib.pyplot as plt

labels = sorted(results['difficulty_label'].unique())
pre_scores = [results[results['difficulty_label']==l]['pre_correct'].mean() for l in labels]
post_scores = [results[results['difficulty_label']==l]['post_correct'].mean() for l in labels]

x = range(len(labels))
width = 0.35
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax1 = axes[0]
ax1.bar([i-width/2 for i in x], pre_scores, width, label='Pre-learning', color='#4C72B0')
ax1.bar([i+width/2 for i in x], post_scores, width, label='Post-learning', color='#55A868')
ax1.set_ylabel('Accuracy')
ax1.set_title('Pre vs Post-Learning')
ax1.set_xticks(list(x))
ax1.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax1.legend()
ax1.set_ylim(0, 1.05)

ax2 = axes[1]
gains = [p-b for b,p in zip(pre_scores, post_scores)]
colors = ['#55A868' if g>=0 else '#C44E52' for g in gains]
ax2.bar(range(len(labels)), gains, color=colors)
ax2.set_ylabel('Learning Gain')
ax2.set_title('Learning Gain by Difficulty')
ax2.set_xticks(list(x))
ax2.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax2.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.savefig('belief_revision_results.png', dpi=150, bbox_inches='tight')
plt.show()
